# 📊 MESA Benchmark Suite — Zero-Cost Colab Mode

Bu notebook, **MESA Benchmark Sistemini (v0.5.1+)** Google Colab'in ücretsiz altyapısı üzerinde, **Ollama** kullanarak sıfır maliyetle test etmenizi sağlar.

---
## 📖 KULLANIM KILAVUZU & OLASI HATALAR (USER MANUAL)

Bu sistemi hatasız çalıştırmak için aşağıdaki uyarılara dikkat etmelisiniz:

### 1. Gerekli LLM (Büyük Dil Modelleri)
- **LLM Yargıcı (LLM Judge):** Sistem, verilen cevapların kalitesini ölçmek için yerel bir LLM Yargıcına ihtiyaç duyar. Bu notebook'ta varsayılan olarak Colab T4 GPU'larına en uygun ve en hızlı olan **`llama3.2:3b`** modeli kullanılmaktadır.
- **Daha Yüksek Başarı İçin:** Eğer Colab Pro kullanıyorsanız, 2. Hücredeki modeli `llama3.1:8b` veya `gemma2:9b` olarak değiştirebilirsiniz.
- **Embedding (Vektör) Modeli:** Otomatik olarak `sentence-transformers/all-MiniLM-L6-v2` kullanılacaktır.

### 2. Kritik Kimlik Bilgileri (Credentials)
- **HF_TOKEN (HuggingFace):** Embedding modeli indirilirken rate-limit (hız sınırı) yememek için Colab'in sol menüsündeki 🔑 (Secrets) sekmesine `HF_TOKEN` eklemeniz tavsiye edilir. (Zorunlu değil ama önerilir).
- **OPENAI_API_KEY:** Bu notebook Ollama kullandığı için sahte bir (dummy) key olan `ollama` atanmıştır. Herhangi bir OpenAI bakiyesine ihtiyacınız yoktur!

### 3. Sık Karşılaşılan Hatalar ve Çözümleri
- ❌ **Hata:** `No such file or directory: 'mesa-benchmark'`
  - **Çözüm:** 1. Hücreyi art arda iki kez çalıştırdığınızda dizin (directory) kayması olur. Menüden `Runtime -> Restart Runtime` yapın ve \"Run All\" tuşuna sadece bir kez basın.\n",
- ❌ **Hata:** `Ollama API Connection Refused`
  - **Çözüm:** 2. Hücredeki Ollama sunucusunun tam olarak ayağa kalkmasını bekleyin. Arka planda sunucunun başlaması 5-10 saniye sürebilir.
- ❌ **Hata:** `CUDA Out of Memory (OOM)`
  - **Çözüm:** Ücretsiz T4 GPU 15GB VRAM'e sahiptir. MESA aynı anda hem LanceDB hem de Ollama çalıştırdığı için belleği zorlayabilir. Eğer OOM alırsanız, `Runtime -> Restart Runtime` yaparak kernel'i temizleyip tekrar çalıştırın.
- ❌ **Hata:** `MemoryPurgeError` veya `Database Locked`
  - **Çözüm:** Önceki test yarıda kesildiyse SQLite kilitlenebilir. `!rm -rf /content/MESA/mesa-benchmark/.benchmark_state` ve `!rm -rf /content/mesa_storage` komutlarını çalıştırıp depolamayı sıfırlayın.

---


## 1. MESA ve Bağımlılıkların Kurulumu

In [ ]:
!apt-get update
!apt-get install -y zstd

import os
# Dizin kaymalarını önlemek için ana dizine geç ve eski MESA klasörünü sil
os.chdir('/content')
!rm -rf MESA

# Repoyu indir
!git clone https://github.com/Yasou13/MESA.git

# Ana bağımlılıkları kur (MESA core)
%cd /content/MESA
!pip install -e .

# Benchmark bağımlılıklarını kur
%cd /content/MESA/mesa-benchmark
!pip install -r requirements.txt

# HuggingFace Token'ı Colab Secrets'tan al (varsa)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        print("✅ HF_TOKEN başarıyla yüklendi.")
except Exception:
    print("⚠️ HF_TOKEN bulunamadı. Anonim (rate-limited) indirme yapılacak.")


## 2. Ollama Sunucusunu Başlatma
LLM Yargıcı (LLM Judge) için `llama3.2:3b` modelini indiriyoruz. OpenAI kütüphanesi bu yerel sunucuya yönlendirilecektir.

In [ ]:
import os
import time
import subprocess

# Dizinimizin mesa-benchmark olduğundan emin olalım
os.chdir('/content/MESA/mesa-benchmark')

# OpenAI API'sini Ollama'nın yerel endpoint'ine yönlendir (Sıfır Maliyet)
os.environ["OPENAI_BASE_URL"] = "http://127.0.0.1:11434/v1"
os.environ["OPENAI_API_KEY"] = "ollama-dummy-key"

# Colab diskini kullan
os.environ["MESA_STORAGE_PATH"] = "/content/mesa_storage"

!curl -fsSL https://ollama.com/install.sh | sh

# Ollama'yı arka planda çalıştır
!nohup ollama serve > /dev/null 2>&1 &

# Sunucunun açılmasını bekle (Sağlamlaştırma)
print("⏳ Ollama sunucusu bekleniyor...")
for _ in range(15):
    try:
        res = subprocess.run(["curl", "-s", "http://127.0.0.1:11434/api/tags"], capture_output=True, text=True)
        if "models" in res.stdout:
            print("✅ Ollama sunucusu aktif!")
            break
    except:
        pass
    time.sleep(2)
else:
    print("❌ Ollama sunucusu başlatılamadı. Runtime'ı yeniden başlatın.")

# Modeli indir (LLM Yargıcı olarak kullanılacak)
!ollama pull llama3.2:3b
print("✅ llama3.2:3b modeli hazır!")

## 3. Benchmark Konfigürasyonu
`config.yaml` dosyasını otomatik güncelleyerek sistemin Ollama'yı yargıç olarak tanımasını sağlıyoruz.

In [ ]:
import yaml
import os

os.chdir('/content/MESA/mesa-benchmark')
config_path = 'config.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# LLM Judge'ı Ollama modelimize ayarla
config['evaluation']['llm_judge_model'] = 'llama3.2:3b'

# Colab'da testin çok uzun sürmemesi için iteration sayısını 1 yapıyoruz
config['iterations'] = 1
if 'client' in config:
    config['client']['retry_count'] = 1  # Timeout anında çok beklememesi için

with open(config_path, 'w') as f:
    yaml.dump(config, f)

print("✅ config.yaml başarıyla güncellendi. Sistem teste hazır!")

## 4. Benchmark'ı Başlat (Run)
Tüm altyapı hazır. Sentetik veri seti izole bir ortamda test edilecek.

In [ ]:
import os
os.chdir('/content/MESA/mesa-benchmark')

# Varsa önceki test kalıntılarını temizle
!rm -rf /content/MESA/mesa-benchmark/.benchmark_state
!rm -rf /content/mesa_storage

# Benchmark'ı çalıştır
!python -m mesa_benchmark

## 5. Raporu Görüntüle (Report)
Test bittiğinde üretilen Markdown raporunu aşağıda görebilirsiniz.

In [ ]:
import os
import glob
from IPython.display import Markdown, display

os.chdir('/content/MESA/mesa-benchmark')

# En son oluşturulan raporu bul ve göster
reports = glob.glob('reports/*.md')
if reports:
    latest_report = max(reports, key=os.path.getctime)
    print(f"📌 Okunan Rapor: {latest_report}\n")
    with open(latest_report, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print("⚠️ Rapor bulunamadı. Test başarısız olmuş olabilir.")